# MATPOWER grid-viz

> conda: f-grid-viz
> /home/isatkaus/flash-w-dom/conda-envs/h-py312-basic/bin/python

> notebook for loading MATPOWER `.m` files into DataFrames using `m_viz_utils.py`.

> load one case, normalize `bus` / `gen` / `branch`, validate table integrity, and preview the data before adding geo utilities and plots.

> plot with plotly


In [1]:
from pathlib import Path
import importlib
import os

import pandas as pd

import m_viz_utils
from m_viz_utils import read_matpower_case, summarize_case, validation_report_df

## environment and display setup

Keep the notebook lightweight for now. We only need enough setup to inspect the `.m` case tables cleanly.

In [2]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

onkestrel = "NREL_CLUSTER" in os.environ and os.environ["NREL_CLUSTER"] == "kestrel"
print(f"onkestrel: {onkestrel}")

onkestrel: True


## configure input paths

Start with one raw TAMU MATPOWER case. Geo input is included only as a placeholder for the next phase.

In [3]:
if onkestrel:
    scidac_data_dir = Path("/kfs2/projects/scidac/scidac-data")
else:
    scidac_data_dir = Path("/home/isatkaus/projects/scidac/scidac-data")

case_name = "ACTIVSg200"
# case_name = "Hawaii40"

# Optional explicit override. Keep None for normal case_name-based path resolution.
# raw_tamu_data_dir_override = Path(
#     "/home/isatkaus/projects/scidac/scidac-data/Hawaii40/raw-tamu-data"
# )
raw_tamu_data_dir_override = None
raw_tamu_data_dir = (
    raw_tamu_data_dir_override
    if raw_tamu_data_dir_override is not None and raw_tamu_data_dir_override.exists()
    else scidac_data_dir / case_name / "raw-tamu-data"
)

# Set names to None for automatic detection from selected case_name.
m_file_name = None
# Optional explicit example for Hawaii custom naming:
# m_file_name = "Hawaii40_20231026.m"
gic_file_name = f"{case_name}_GIC_data.gic"
aux_file_name = None

if m_file_name is None:
    default_m = raw_tamu_data_dir / f"case_{case_name}.m"
    if default_m.exists():
        m_file_path = default_m
    else:
        m_candidates = sorted(raw_tamu_data_dir.glob("*.m"))
        if len(m_candidates) == 1:
            m_file_path = m_candidates[0]
        else:
            raise FileNotFoundError(
                f"Could not uniquely determine .m file in {raw_tamu_data_dir}. "
                f"Candidates: {[p.name for p in m_candidates]}"
            )
else:
    m_file_path = raw_tamu_data_dir / m_file_name

gic_file_path = raw_tamu_data_dir / gic_file_name
if aux_file_name is None:
    default_aux = m_file_path.with_suffix(".AUX")
    if default_aux.exists():
        aux_file_path = default_aux
    else:
        aux_file_path = raw_tamu_data_dir / f"{case_name}.AUX"
else:
    aux_file_path = raw_tamu_data_dir / aux_file_name

geo_file_path = gic_file_path if gic_file_path.exists() else aux_file_path

if not m_file_path.exists():
    raise FileNotFoundError(f"MATPOWER .m file not found: {m_file_path}")
if not gic_file_path.exists():
    if aux_file_path.exists():
        print(f"WARNING: GIC file not found at {gic_file_path}")
        print(f"INFO: proceeding with AUX for geo plotting: {aux_file_path}")
    else:
        print(f"WARNING: GIC file not found at {gic_file_path}")
if not aux_file_path.exists():
    print(f"WARNING: AUX file not found at {aux_file_path}")
if not geo_file_path.exists():
    print("WARNING: No geo source file found (.gic or .AUX)")

print(f"raw_tamu_data_dir: {raw_tamu_data_dir}")
print(f"m_file_path:       {m_file_path}")
print(f"gic_file_path:     {gic_file_path}")
print(f"aux_file_path:     {aux_file_path}")
print(f"geo_file_path:     {geo_file_path}")

raw_tamu_data_dir: /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data
m_file_path:       /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/case_ACTIVSg200.m
gic_file_path:     /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/ACTIVSg200_GIC_data.gic
aux_file_path:     /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/ACTIVSg200.AUX
geo_file_path:     /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/ACTIVSg200_GIC_data.gic


## load MATPOWER case

Read the `.m` file into normalized `bus` / `gen` / `branch` tables via `m_viz_utils.py`.

In [4]:
importlib.reload(m_viz_utils)
from m_viz_utils import read_matpower_case, summarize_case, validation_report_df

case_data = read_matpower_case(m_file_path)
summary = summarize_case(case_data)
validation_df = validation_report_df(case_data)

summary

<module 'm_viz_utils' from '/kfs2/projects/scidac/isatkaus/scidac-notebooks/m_viz_utils.py'>

case_name                                                                                  case_ACTIVSg200
source_path                   /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/case_ACTIVSg200.m
base_mva                                                                                             100.0
version                                                                                                  2
n_buses                                                                                                200
n_generators                                                                                            49
n_branches                                                                                             245
n_unique_gen_buses                                                                                      49
n_unique_branch_from_buses                                                                             174
n_unique_branch_to_buses             

## validation

These checks confirm that generator and branch bus references map back to valid `BUS_I` values.

In [5]:
validation_df

,check,value
0,required_columns,"{'bus_has_BUS_I': True, 'gen_has_GEN_BUS': True, 'branch_has_F_BUS': True, '..."
1,all_required_columns_present,True
2,missing_gen_bus_ids,[]
3,missing_branch_from_bus_ids,[]
4,missing_branch_to_bus_ids,[]
5,gen_buses_are_valid,True
6,branch_from_buses_are_valid,True
7,branch_to_buses_are_valid,True


## quick previews

Preview the normalized tables before adding geo joins or plotting logic.

In [6]:
print("bus table")
case_data.bus.head()

print("gen table")
case_data.gen.head()

print("branch table")
case_data.branch.head()

bus table


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN,LAM_P,LAM_Q,MU_VMAX,MU_VMIN,bus_name
BUS_I,,,,,,,,,,,,,,,,,,
1,1,1.0,0.00,0.00,0.0,0.0,1.0,1.019152,-7.085196,115.0,2.0,1.1,0.9,6.87,0.0,0.0,0.0,CREVE COEUR 0
2,2,1.0,7.39,2.10,0.0,0.0,1.0,1.019035,-7.098018,115.0,2.0,1.1,0.9,6.87,0.0,0.0,0.0,CREVE COEUR 1
3,3,1.0,0.00,0.00,0.0,0.0,1.0,1.030054,-10.029102,115.0,4.0,1.1,0.9,6.87,0.0,0.0,0.0,ILLIOPOLIS 0
4,4,1.0,1.70,0.48,0.0,0.0,1.0,1.030029,-10.031977,115.0,4.0,1.1,0.9,6.87,0.0,0.0,0.0,ILLIOPOLIS 1
5,5,1.0,0.00,0.00,0.0,0.0,1.0,1.037273,-3.546274,115.0,6.0,1.1,0.9,6.87,0.0,0.0,0.0,PAXTON 2 0


gen table


,gen_row,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,PC1,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF,MU_PMAX,MU_PMIN,MU_QMAX,MU_QMIN
gen_row,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,49,1.36,0.88,2.11,-0.55,1.04,5.44,1.0,4.53,1.36,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,50,1.36,1.18,2.11,-0.55,1.04,5.44,1.0,4.53,1.36,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,51,1.36,0.77,2.11,-0.55,1.04,5.44,1.0,4.53,1.36,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,52,1.36,1.25,2.11,-0.55,1.04,5.44,1.0,4.53,1.36,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,5,53,2.72,0.79,4.23,-1.11,1.04,10.88,1.0,9.07,2.72,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


branch table


,branch_row,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT,MU_SF,MU_ST,MU_ANGMIN,MU_ANGMAX
branch_row,,,,,,,,,,,,,,,,,,,,,,
1,1,2,1,0.000673,0.003339,0.00000,100.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-7.39,-2.10,7.39,2.11,0.0,0.0,0.0,0.0
2,2,1,119,0.018542,0.119758,0.02285,221.1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,12.25,-0.57,-12.22,-1.63,0.0,0.0,0.0,0.0
3,3,124,1,0.005615,0.036268,0.00692,221.1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,40.13,7.94,-40.04,-8.08,0.0,0.0,0.0,0.0
4,4,193,1,0.004258,0.027501,0.00525,221.1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-20.39,-6.96,20.41,6.54,0.0,0.0,0.0,0.0
5,5,4,3,0.000573,0.003303,0.00000,100.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-1.70,-0.48,1.70,0.48,0.0,0.0,0.0,0.0


## optional auxiliary tables

If the MATPOWER file exposes generator fuel labels or cost tables, inspect them here.

In [7]:
if case_data.genfuel is not None:
    print("genfuel index")
    case_data.genfuel

if case_data.gencost is not None:
    print("gencost table")
    case_data.gencost.head()

genfuel index


Index(['coal', 'coal', 'coal', 'coal', 'coal', 'wind', 'coal', 'coal', 'coal',
       'coal', 'coal', 'coal', 'coal', 'ng', 'ng', 'ng', 'ng', 'ng', 'ng',
       'ng', 'coal', 'wind', 'wind', 'wind', 'wind', 'coal', 'coal', 'coal',
       'coal', 'coal', 'wind', 'coal', 'coal', 'coal', 'coal', 'coal', 'ng',
       'ng', 'ng', 'ng', 'ng', 'ng', 'ng', 'ng', 'coal', 'coal', 'nuclear',
       'ng', 'ng'],
      dtype='str', name='genfuel')

gencost table


,MODEL,STARTUP,SHUTDOWN,NCOST,C2,C1,C0
gen,,,,,,,
1,2.0,0.0,0.0,3.0,0.002,19.0,236.12
2,2.0,0.0,0.0,3.0,0.002,19.0,236.12
3,2.0,0.0,0.0,3.0,0.002,19.0,236.12
4,2.0,0.0,0.0,3.0,0.002,19.0,236.12
5,2.0,0.0,0.0,3.0,0.002,19.0,236.24


## next phase hooks

The next iteration will keep using this notebook and add:

1. GIC parsing into bus coordinates.
2. Split-point handling for colocated buses.
3. Geo propagation to generator and branch tables.
4. Plotly map-based plotting similar to `pcm_viz.ipynb`.

In [8]:
defer_geo_merge = True
print(f"defer_geo_merge: {defer_geo_merge}")
print("Geo merge and plotting are intentionally deferred in this first implementation.")

defer_geo_merge: True
Geo merge and plotting are intentionally deferred in this first implementation.


## geo merge and split points

Load bus coordinates from a geo source file (`.gic` or `.AUX`), merge into the MATPOWER tables, and split colocated bus points before plotting.

For Hawaii, generator markers are also fanned out radially around each bus (visual-only) so overlapping generators remain individually hoverable.

In [9]:
importlib.reload(m_viz_utils)
import plotly.graph_objects as go
from m_viz_utils import attach_geo_to_case

enable_hawaii_gen_fanout = "hawaii" in str(case_name).lower()
gen_fanout_radius = 0.0025

if geo_file_path.exists():
    geo_result = attach_geo_to_case(
        case_data,
        geo_file_path,
        split_colocated=True,
        split_radius=0.006,
        gen_fanout=enable_hawaii_gen_fanout,
        gen_fanout_radius=gen_fanout_radius,
    )
    geo_case_data = geo_result.case_data

    # --- optional: attach GridKit JSON ids to hover text ---
    # Set case_json_path to the corresponding GridKit JSON case file, or None to skip.
    import gridkit_utils

    importlib.reload(gridkit_utils)
    from gridkit_utils import attach_json_ids

    _json_candidates = {
        "Hawaii40": Path(
            "/home/isatkaus/install/gridkit-manual/GridKit/examples/PhasorDynamics/Medium/Hawaii/hawaii.json"
        ),
        "ACTIVSg200": Path(
            "/home/isatkaus/install/gridkit-manual/GridKit/examples/PhasorDynamics/Large/Illinois/illinois.json"
        ),
    }
    case_json_path = _json_candidates.get(case_name, None)
    if case_json_path is not None and case_json_path.exists():
        attach_json_ids(geo_case_data, case_json_path)
        print(f"JSON ids attached from {case_json_path.name}")
    else:
        print("JSON ids not attached (no matching case_json_path)")

    n_multi_gen_buses = int((case_data.gen.groupby("GEN_BUS").size() > 1).sum())
    pd.Series(
        {
            "geo_source": str(geo_file_path),
            "split_applied": geo_result.split_applied,
            "gen_fanout_enabled": enable_hawaii_gen_fanout,
            "gen_fanout_radius": gen_fanout_radius,
            "n_buses_with_multiple_gens": n_multi_gen_buses,
            "unique_bus_locations_before": geo_result.n_unique_bus_locations_before,
            "unique_bus_locations_after": geo_result.n_unique_bus_locations_after,
        }
    )
else:
    geo_result = None
    geo_case_data = None
    print(
        "Geo prep skipped: no geo source file found. "
        "Set geo_file_path to a valid .gic or .AUX file, then rerun this cell."
    )

<module 'm_viz_utils' from '/kfs2/projects/scidac/isatkaus/scidac-notebooks/m_viz_utils.py'>

<module 'gridkit_utils' from '/kfs2/projects/scidac/isatkaus/scidac-notebooks/gridkit_utils.py'>

JSON ids attached from illinois.json


geo_source                     /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/ACTIVSg200_GIC_da...
split_applied                                                                                             True
gen_fanout_enabled                                                                                       False
gen_fanout_radius                                                                                       0.0025
n_buses_with_multiple_gens                                                                                   0
unique_bus_locations_before                                                                                111
unique_bus_locations_after                                                                                 200
dtype: object

## geo plot

Initial map with branches, buses (sized by PD), and generators (fuel-colored when available).

In [10]:
importlib.reload(m_viz_utils)
from m_viz_utils import plot_grid, lookup_fault_bus

# --- plot options ---
show_loading = (
    True  # True: viridis-colored branches by loading_pct (requires PF/QF/PT/QT/RATE_A)
)
# show_loading = False  # False: uniform gray branches

# save_html = False  # True: write figure to save_figs_dir as .html
save_html = True

save_figs_dir = scidac_data_dir / "figs"  # adjust as needed
# save_figs_dir = Path("/home/isatkaus/projects/scidac/isatkaus/scidac-notebooks/figs")

show_gen_connectors = "hawaii" in str(case_name).lower()

# --- figure size (pixels; None = use plot_grid defaults: width=780, height=650) ---
fig_width = 720
fig_height = None

# --- fault bus (optional) ---
# Identify a faulted bus by BUS_I (int) or bus_name (str, from mpc.bus_name).
# Set to None to skip.
# fault_bus = 1             # ACTIVSg200: by BUS_I integer (parametric — all 200 buses wired)
# fault_bus = "ALOHA138"    # Hawaii40: by bus_name string (bus 1, fixed fault)
_fault_defaults = {
    "Hawaii40": "ALOHA138",
    "ACTIVSg200": None,  # parametric — no single canonical fault bus
}
fault_bus = _fault_defaults.get(case_name, None)

if geo_case_data is not None:
    fig_geo = plot_grid(
        geo_case_data,
        zoom=7,
        show_loading=show_loading,
        show_gen_connectors=show_gen_connectors,
    )
    if fig_width is not None or fig_height is not None:
        _ = fig_geo.update_layout(
            width=fig_width if fig_width is not None else fig_geo.layout.width,
            height=fig_height if fig_height is not None else fig_geo.layout.height,
        )

    # --- fault bus overlay ---
    if fault_bus is not None:
        bus_row = lookup_fault_bus(geo_case_data.bus, fault_bus)
        if bus_row.empty:
            bus_df = geo_case_data.bus
            print(
                f"WARNING: fault_bus={fault_bus!r} not found.\n"
                f"Available bus_name values: {bus_df['bus_name'].tolist() if 'bus_name' in bus_df.columns else 'N/A'}\n"
                f"BUS_I range: {bus_df['BUS_I'].min()} – {bus_df['BUS_I'].max()}"
            )
        else:
            fault_lat = float(bus_row["lat"].iloc[0])
            fault_lon = float(bus_row["lon"].iloc[0])
            fault_bus_i = int(bus_row["BUS_I"].iloc[0])
            fault_label = (
                bus_row["bus_name"].iloc[0]
                if "bus_name" in bus_row.columns
                else str(fault_bus_i)
            )
            fault_group = f"fault_{fault_bus_i}"

            _ = fig_geo.add_trace(
                go.Scattermap(
                    mode="markers",
                    lat=[fault_lat],
                    lon=[fault_lon],
                    hoverinfo="skip",
                    marker=dict(size=36, color="black", opacity=1.0),
                    legendgroup=fault_group,
                    showlegend=False,
                )
            )
            _ = fig_geo.add_trace(
                go.Scattermap(
                    name=f"⚡ fault: {fault_label}",
                    mode="markers",
                    lat=[fault_lat],
                    lon=[fault_lon],
                    hovertext=[
                        f"<b>⚡ FAULT</b><br>BUS_I: {fault_bus_i}<br>Name: {fault_label}"
                    ],
                    hoverinfo="text",
                    marker=dict(size=26, color="red", opacity=1.0),
                    legendgroup=fault_group,
                    showlegend=True,
                )
            )

    if save_html:
        save_figs_dir.mkdir(parents=True, exist_ok=True)
        suffix = "_loading" if show_loading else ""
        fault_suffix = f"_fault{fault_bus}" if fault_bus is not None else ""
        html_stem = f"{case_name}_geo{suffix}{fault_suffix}"
        html_path = save_figs_dir / f"{html_stem}.html"
        # height=None + viewport CSS → fills browser window in new tab;
        # fits the iframe height in the MkDocs page without internal scrollbars
        fig_html = go.Figure(fig_geo)
        _ = fig_html.update_layout(height=None, width=None, autosize=True)
        _viewport_css = (
            "var s=document.createElement('style');"
            "s.textContent='html,body{height:100vh;margin:0;padding:0;overflow:hidden;}';"
            "document.head.appendChild(s);"
        )
        fig_html.write_html(
            str(html_path),
            full_html=True,
            include_plotlyjs=True,
            post_script=_viewport_css,
        )
        print(f"figure saved to {html_path}")

    fig_geo
else:
    print(
        "Geo plot skipped because geo_case_data is not available. "
        "Provide a valid geo_file_path and rerun geo prep cell first."
    )

<module 'm_viz_utils' from '/kfs2/projects/scidac/isatkaus/scidac-notebooks/m_viz_utils.py'>

figure saved to /kfs2/projects/scidac/scidac-data/figs/ACTIVSg200_geo_loading.html


In [ ]:
# importlib.reload(m_viz_utils)
# from m_viz_utils import plot_grid, lookup_fault_bus

# # --- plot options ---
# show_loading = (
#     True  # True: viridis-colored branches by loading_pct (requires PF/QF/PT/QT/RATE_A)
# )
# # show_loading = False  # False: uniform gray branches

# # save_html = False  # True: write figure to save_figs_dir as .html
# save_html = True

# save_figs_dir = scidac_data_dir / "figs"  # adjust as needed
# # save_figs_dir = Path("/home/isatkaus/projects/scidac/isatkaus/scidac-notebooks/figs")

# show_gen_connectors = "hawaii" in str(case_name).lower()

# # --- fault bus (optional) ---
# # Identify a faulted bus by BUS_I (int) or bus_name (str, from mpc.bus_name).
# # Set to None to skip.
# fault_bus = None
# # fault_bus = 1           # ACTIVSg200: by BUS_I integer
# fault_bus = "ALOHA138"  # Hawaii40: by bus_name string

# if geo_case_data is not None:
#     fig_geo = plot_grid(
#         geo_case_data,
#         zoom=7,
#         show_loading=show_loading,
#         show_gen_connectors=show_gen_connectors,
#     )

#     # --- fault bus overlay ---
#     if fault_bus is not None:
#         bus_row = lookup_fault_bus(geo_case_data.bus, fault_bus)
#         if bus_row.empty:
#             bus_df = geo_case_data.bus
#             print(
#                 f"WARNING: fault_bus={fault_bus!r} not found.\n"
#                 f"Available bus_name values: {bus_df['bus_name'].tolist() if 'bus_name' in bus_df.columns else 'N/A'}\n"
#                 f"BUS_I range: {bus_df['BUS_I'].min()} – {bus_df['BUS_I'].max()}"
#             )
#         else:
#             fault_lat = float(bus_row["lat"].iloc[0])
#             fault_lon = float(bus_row["lon"].iloc[0])
#             fault_bus_i = int(bus_row["BUS_I"].iloc[0])
#             fault_label = (
#                 bus_row["bus_name"].iloc[0]
#                 if "bus_name" in bus_row.columns
#                 else str(fault_bus_i)
#             )
#             fault_group = f"fault_{fault_bus_i}"

#             _ = fig_geo.add_trace(
#                 go.Scattermap(
#                     mode="markers",
#                     lat=[fault_lat],
#                     lon=[fault_lon],
#                     hoverinfo="skip",
#                     marker=dict(size=36, color="black", opacity=1.0),
#                     legendgroup=fault_group,
#                     showlegend=False,
#                 )
#             )
#             _ = fig_geo.add_trace(
#                 go.Scattermap(
#                     name=f"⚡ fault: {fault_label}",
#                     mode="markers",
#                     lat=[fault_lat],
#                     lon=[fault_lon],
#                     hovertext=[
#                         f"<b>⚡ FAULT</b><br>BUS_I: {fault_bus_i}<br>Name: {fault_label}"
#                     ],
#                     hoverinfo="text",
#                     marker=dict(size=26, color="red", opacity=1.0),
#                     legendgroup=fault_group,
#                     showlegend=True,
#                 )
#             )

#     if save_html:
#         save_figs_dir.mkdir(parents=True, exist_ok=True)
#         suffix = "_loading" if show_loading else ""
#         fault_suffix = f"_fault{fault_bus}" if fault_bus is not None else ""
#         html_stem = f"{case_name}_geo{suffix}{fault_suffix}"
#         html_path = save_figs_dir / f"{html_stem}.html"
#         # height=None → 100% of container; fills browser window in new tab,
#         # fits the iframe height in the MkDocs page without internal scrollbars
#         fig_html = go.Figure(fig_geo)
#         _ = fig_html.update_layout(height=None)
#         fig_html.write_html(str(html_path), full_html=True, include_plotlyjs=True)
#         print(f"figure saved to {html_path}")

#     fig_geo
# else:
#     print(
#         "Geo plot skipped because geo_case_data is not available. "
#         "Provide a valid geo_file_path and rerun geo prep cell first."
#     )

In [ ]:
show_gen_connectors = "hawaii" in str(case_name).lower()

# --- figure size (pixels; None = use plot_grid defaults: width=780, height=650) ---
fig_width = None
fig_height = None